In [1]:
import os

# Replace with the actual address of your Prefect server
#os.environ["PREFECT_API_URL"] = "http://prefect.local/api"
#os.environ["PREFECT_API_URL"] = "http://10.43.153.8:4200/api"
#os.environ["PREFECT_API_URL"] = "http://10.2.97.207:4200/api"

In [2]:
from dask_kubernetes.operator import KubeCluster 
from dask.distributed import Client 
import dask.array as da 
import time
import os

In [3]:
import dask
import dask.distributed
from prefect_dask.task_runners import DaskTaskRunner

/opt/conda/lib/python3.12/site-packages/tzlocal/unix.py:207: UserWarning: Can not find any timezone configuration, defaulting to UTC.
  warnings.warn("Can not find any timezone configuration, defaulting to UTC.")


In [4]:
from dask_kubernetes.operator import KubeCluster, make_cluster_spec
import os

username = os.environ["JUPYTERHUB_USER"]

spec = make_cluster_spec(
    name=f"manual-dask-cluster-{username}",
    image="ghcr.io/casangi/radps-dask-worker",
    worker_command=[
        "dask-worker",
        "--nworkers", "4",
        "--nthreads", "1",
        "--memory-limit", "4G",
    ],
    scheduler_service_type="NodePort"
)

worker_spec = spec["spec"]["worker"]["spec"]

# Add volume
worker_spec.setdefault("volumes", []).append(
    {
        "name": "shared-storage",
        "persistentVolumeClaim": {
            "claimName": "radps-hub-pvc"
        }
    }
)

# Mount volume in the worker container
worker_container = worker_spec["containers"][0]
worker_container.setdefault("volumeMounts", []).append(
    {
        "name": "shared-storage",
        "mountPath": "/home/jovyan/shared"
    }
)

# Permissions fix
worker_spec.setdefault("securityContext", {})["fsGroup"] = 100

# Restrict workers to Longhorn-capable nodes (exclude ework nodes which have no Longhorn storage)
worker_spec.setdefault("affinity", {})["nodeAffinity"] = {
    "requiredDuringSchedulingIgnoredDuringExecution": {
        "nodeSelectorTerms": [
            {
                "matchExpressions": [
                    {
                        "key": "kubernetes.io/hostname",
                        "operator": "NotIn",
                        "values": [
                            "radps-k3s-ework5",
                            "radps-k3s-ework6",
                            "radps-k3s-ework7",
                            "radps-k3s-ework8",
                        ],
                    }
                ]
            }
        ]
    }
}

cluster = KubeCluster(
    custom_cluster_spec=spec,
    namespace="radps-hub",
)

Output()

In [5]:
# Scale the cluster to 2 pod 
print("Two pods with 4 workers each")
cluster.scale(2)

# It can take a minute or two for the pods to be created and ready. 
# The following call will block until the workers are available. 
cluster.wait_for_workers(2) 
print("Cluster is ready with 8 workers on 2 pods.")

Two pods with 4 workers each


KeyboardInterrupt: 

In [9]:
!kubectl get pods --namespace radps-hub

NAME                                                              READY   STATUS              RESTARTS      AGE
continuous-image-puller-7f48p                                     1/1     Running             2 (82d ago)   111d
continuous-image-puller-cmg2x                                     1/1     Running             2 (82d ago)   111d
continuous-image-puller-jgqn7                                     1/1     Running             3 (81d ago)   111d
dask-operator-dask-kubernetes-operator-555955b666-mxqqq           1/1     Running             0             82d
hub-784f7dfc57-74zsq                                              1/1     Running             0             81d
jupyter-jsteeb                                                    1/1     Running             0             39m
manual-dask-cluster-jsteeb-default-worker-8299846dd0-5d66bw6gm2   0/1     ContainerCreating   0             19m
manual-dask-cluster-jsteeb-default-worker-fdd9e18ee8-cd446qj848   0/1     ContainerCreating   0      

In [10]:
!kubectl get services --namespace radps-hub

NAME                                   TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)                         AGE
hub                                    ClusterIP   10.43.1.78      <none>        8081/TCP                        111d
manual-dask-cluster-jsteeb-scheduler   NodePort    10.43.139.132   <none>        8786:32335/TCP,8787:31359/TCP   19m
prefect-server                         NodePort    10.43.137.247   <none>        4200:30042/TCP                  111d
prefect-server-postgresql              ClusterIP   10.43.6.17      <none>        5432/TCP                        111d
prefect-server-postgresql-hl           ClusterIP   None            <none>        5432/TCP                        111d
proxy-api                              ClusterIP   10.43.172.174   <none>        8001/TCP                        111d
proxy-public                           NodePort    10.43.25.236    <none>        443:30443/TCP,80:31538/TCP      111d


In [ ]:
# Connect a Dask client to the cluster 
client = Client(cluster) 
print("Dask client connected.") 
client

In [ ]:
# Assign Prefect to use Dask Task Runner
task_runner=DaskTaskRunner(client.scheduler)

In [ ]:
# Example workflow

from prefect import flow, task
from prefect_dask.task_runners import DaskTaskRunner
import dask
import time

@task
def image_channel_chunk():
    import time
    time.sleep(0.1)
    return 42

@task
def image_single_spw(x):
    time.sleep(1)

    n_cc = 1000
    return_vals_list = []
    for i_cc in range(n_cc):
        delayed_return_val = dask.delayed(image_channel_chunk.fn)()  # use .fn to get raw callable
        return_vals_list.append(delayed_return_val)
        
    dask.compute(*return_vals_list)
        
    return x**2

task_runner=DaskTaskRunner(address=client.scheduler.address)

@flow(task_runner=task_runner)
def image_cube(n_spw):
    futures = [image_single_spw.submit(i) for i in range(n_spw)]
    return [f.result() for f in futures]

image_cube(n_spw=16)

In [6]:
# Important to run otherwise zombie processes happen.
client.close()
cluster.close()

NameError: name 'client' is not defined

In [ ]:
k get storageclass